In [ ]:
# ===== v3: ALIAS-AWARE + DETERMINISTIC crosscheck (see Cell 6b; ORDER BY in joins) =====
# Cell 1 — imports
import re
import redivis
import pandas as pd

MIN_COMPANY_NAME_LEN = 4

In [ ]:
# Cell 2 — load our URL list from the v2 dataset (only need linkedin_url for SQL JOIN)
urls_table = redivis.user("ml2068").dataset("all_linkedin_urls_v2:ce7f").table("all_linkedin_urls_v2:0wzc")
urls_df = urls_table.to_pandas_dataframe(variables=["linkedin_url"])

def _clean(u):
    if u is None or pd.isna(u):
        return None
    s = re.sub(r"^https?://(www\.)?", "", str(u).strip().rstrip("/"))
    return s if s.startswith("linkedin.com/in/") else None

urls_df["clean_linkedin_url"] = urls_df["linkedin_url"].apply(_clean)
urls_df = urls_df[urls_df["clean_linkedin_url"].notna()]
print(f"Our URLs: {len(urls_df):,} rows")
urls_df.head(3)

In [ ]:
# Cell 3 — join URLs → individual_user (server-side, only matched rows returned)
matched_users_df = redivis.query("""
    SELECT
        u.user_id, u.firstname, u.lastname, u.fullname,
        u.profile_linkedin_url, u.profile_title, u.numconnections,
        u.user_country, u.prestige,
        REGEXP_REPLACE(REGEXP_REPLACE(urls.linkedin_url, r'^https?://(www\\.)?', ''), r'/$', '') AS clean_linkedin_url
    FROM `all_linkedin_urls_v2:0wzc` AS urls
    INNER JOIN `individual_user:xcsm` AS u
        ON REGEXP_REPLACE(REGEXP_REPLACE(urls.linkedin_url, r'^https?://(www\\.)?', ''), r'/$', '') = u.profile_linkedin_url
    ORDER BY clean_linkedin_url, u.user_id
""").to_pandas_dataframe()

print(f"Matched users: {len(matched_users_df):,} rows")
matched_users_df.head(3)

In [ ]:
# Cell 4 — fetch positions via JOIN (avoids IN clause size limit)
positions_df = redivis.query("""
    SELECT
        p.user_id,
        p.company_cleaned,
        p.seniority,
        p.startdate,
        p.enddate
    FROM `individual_position:8xgp` AS p
    INNER JOIN `matched_users:kh5e` AS m
        ON p.user_id = m.user_id
    ORDER BY p.user_id, p.company_cleaned
""").to_pandas_dataframe()

print(f"Positions fetched: {len(positions_df):,} rows")
positions_df.head(3)

In [ ]:
# Cell 5 — build positions index {user_id: [company_cleaned, ...]}
positions_index = {}
for row in positions_df[["user_id", "company_cleaned"]].itertuples(index=False):
    uid = row.user_id
    if uid not in positions_index:
        positions_index[uid] = []
    positions_index[uid].append(row.company_cleaned)

print(f"Position index built for {len(positions_index):,} users")

In [ ]:
# Cell 6 — helper functions
import unicodedata
from difflib import SequenceMatcher

SUFFIXES = {
    "jr", "sr", "ii", "iii", "iv",
    "phd", "ph.d", "ph.d.", "md", "m.d", "m.d.",
    "jd", "j.d", "j.d.", "mba", "m.b.a", "cpa", "cfa", "esq",
    "ba", "ma", "ms", "bs", "mpa", "mph",
    "icd.d", "icd", "ret", "retired",
    "usaf", "usa", "usmc", "usn", "uscg", "cm",
}

LEGAL_SUFFIXES = re.compile(
    r"\b(inc|llc|corp|corporation|ltd|limited|gmbh|co|group|plc|sa|ag|bv|nv|lp|llp|holdings|holding|international|intl)\b",
    re.IGNORECASE
)


def to_ascii(s):
    """Normalize accents and special chars → plain ASCII lowercase."""
    if pd.isna(s):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKD", s)
    s = s.encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9\s]", "", s.lower()).strip()


def clean_person_name(name):
    """Strip credentials/suffixes and return cleaned name tokens."""
    if pd.isna(name):
        return []
    name = str(name).split(",")[0].strip()
    tokens = name.lower().split()
    while tokens and tokens[-1].rstrip(".") in SUFFIXES:
        tokens.pop()
    return tokens


def name_matches(revelio_fullname, our_name, our_name_clean=None):
    """Four-stage cascade: last name → hyphen → first+initial → difflib."""
    if pd.isna(revelio_fullname):
        return False
    rev = to_ascii(revelio_fullname)

    for name in [our_name_clean, our_name]:
        tokens = clean_person_name(name)
        if not tokens:
            continue
        last = to_ascii(tokens[-1])
        first = to_ascii(tokens[0]) if len(tokens) > 1 else ""

        # Stage 1: last name substring
        if last and last in rev:
            return True

        # Stage 2: hyphenated last name — check each part
        if "-" in last:
            if any(part in rev for part in last.split("-") if part):
                return True

        # Stage 3: first name present + last initial present
        if first and last and first in rev and last[0] in rev:
            return True

        # Stage 4: difflib similarity on full name
        our_full = to_ascii(" ".join(tokens))
        if our_full and SequenceMatcher(None, our_full, rev).ratio() >= 0.85:
            return True

    return False


def strip_legal(name):
    """Strip legal suffixes and normalize to ASCII."""
    return to_ascii(LEGAL_SUFFIXES.sub("", str(name))).strip()


def company_in_positions(user_id, company_name, company_name_orig=None):
    """Bidirectional substring + token overlap after legal suffix stripping."""
    positions = positions_index.get(int(user_id), [])
    for name in [company_name, company_name_orig]:
        if pd.isna(name) or len(str(name)) < MIN_COMPANY_NAME_LEN:
            continue
        our = strip_legal(name)
        our_tokens = set(our.split()) - {""}
        if not our_tokens:
            continue
        for pos in positions:
            if pd.isna(pos):
                continue
            pos_clean = strip_legal(pos)
            # Bidirectional substring
            if our in pos_clean or pos_clean in our:
                return True
            # Token overlap: ≥50% of our tokens appear in position string
            pos_tokens = set(pos_clean.split())
            overlap = our_tokens & pos_tokens
            if len(overlap) / len(our_tokens) >= 0.5:
                return True
    return False


def clean_url(url):
    """Normalise to linkedin.com/in/<slug>."""
    if pd.isna(url):
        return None
    url = str(url).strip().rstrip("/")
    url = re.sub(r"^https?://(www\.)?", "", url)
    return url if url.startswith("linkedin.com/in/") else None


# ──────────────────────────────────────────────────────────
# Fuzzy company matcher — catches parent/subsidiary, abbreviations,
# and short names that the strict matcher misses. Computed in
# PARALLEL with the strict version so we can compare directly.
# ──────────────────────────────────────────────────────────

FUZZY_MIN_COMPANY_LEN = 3   # allow IBM, GE, HP, AT&T (vs strict's 4)
FUZZY_RATIO_THRESHOLD = 0.80


def company_in_positions_fuzzy(user_id, company_name, company_name_orig=None):
    """Strict match first; if that fails, try difflib similarity ≥ 0.80."""
    if company_in_positions(user_id, company_name, company_name_orig):
        return True

    positions = positions_index.get(int(user_id), [])
    for name in [company_name, company_name_orig]:
        if pd.isna(name) or len(str(name)) < FUZZY_MIN_COMPANY_LEN:
            continue
        our = strip_legal(name)
        if not our or len(our) < FUZZY_MIN_COMPANY_LEN:
            continue
        for pos in positions:
            if pd.isna(pos):
                continue
            pos_clean = strip_legal(pos)
            if not pos_clean or len(pos_clean) < FUZZY_MIN_COMPANY_LEN:
                continue
            if SequenceMatcher(None, our, pos_clean).ratio() >= FUZZY_RATIO_THRESHOLD:
                return True
    return False


# ──────────────────────────────────────────────────────────
# v3 ALIAS-AWARE company matchers — test the firm's WHOLE alias
# set (keyed on gvkey, built in Cell 6b) against Revelio positions.
# ──────────────────────────────────────────────────────────

def company_in_positions_aliased(user_id, alias_set):
    positions = positions_index.get(int(user_id), [])
    if not positions or not alias_set:
        return False
    pos_clean = [strip_legal(p) for p in positions if not pd.isna(p)]
    for our in alias_set:
        toks = set(our.split()) - {""}
        if not toks:
            continue
        for pc in pos_clean:
            if not pc:
                continue
            if our in pc or pc in our:
                return True
            if len(toks & set(pc.split())) / len(toks) >= 0.5:
                return True
    return False


def company_in_positions_aliased_fuzzy(user_id, alias_set):
    if company_in_positions_aliased(user_id, alias_set):
        return True
    pos_clean = [strip_legal(p) for p in positions_index.get(int(user_id), []) if not pd.isna(p)]
    for our in alias_set:
        if len(our) < FUZZY_MIN_COMPANY_LEN:
            continue
        for pc in pos_clean:
            if not pc or len(pc) < FUZZY_MIN_COMPANY_LEN:
                continue
            if SequenceMatcher(None, our, pc).ratio() >= FUZZY_RATIO_THRESHOLD:
                return True
    return False


In [ ]:
# Cell 6b — load company aliases -> {gvkey: set(normalized aliases)}.  [v3 NEW]
# STEP 1: upload data/revelio/company_aliases.csv to your Redivis ml2068 workspace as a
# table, then put its version tag in <ver> below. Non-date-gated (lenient).

def norm_gvkey(x):
    try:
        return str(int(float(x)))
    except (ValueError, TypeError):
        return None

aliases_df = redivis.user("ml2068").dataset("company_aliases:<ver>").table("company_aliases").to_pandas_dataframe(variables=["gvkey", "alias_name_clean"])
# Fallback if attached as a notebook file: aliases_df = pd.read_csv("company_aliases.csv", usecols=["gvkey","alias_name_clean"])

alias_by_gvkey = {}
for r in aliases_df.itertuples(index=False):
    g = norm_gvkey(r.gvkey)
    if g is None:
        continue
    a = strip_legal(r.alias_name_clean)
    if a and len(a) >= MIN_COMPANY_NAME_LEN:
        alias_by_gvkey.setdefault(g, set()).add(a)

print(f"alias sets: {len(alias_by_gvkey):,} gvkeys")
print("sanity - Alphabet(160329):", sorted(alias_by_gvkey.get("160329", [])))  # expect ['alphabet','google']


In [ ]:
# Cell 7 — build lookup: clean_url → revelio user row
revelio_by_url = matched_users_df.sort_values(["clean_linkedin_url", "user_id"]).drop_duplicates("clean_linkedin_url", keep="first").set_index("clean_linkedin_url")

In [ ]:
# Cell 8 — load all_linkedin_urls v2 (with board_company + primary_company)
# v2 schema: separate board_company (WRDS) and primary_company (DEF 14A) so we can
# check Revelio work history against EITHER candidate per row.
all_urls_table = redivis.user("ml2068").dataset("all_linkedin_urls_v2:ce7f").table("all_linkedin_urls_v2:0wzc")
all_people_df = all_urls_table.to_pandas_dataframe(
    variables=["person_name", "person_name_clean",
               "board_company", "primary_company", "company_name_clean",
               "source", "linkedin_url", "verified",
               "gvkey", "ticker", "search_anchor_used"]
)
print(f"all_linkedin_urls v2: {len(all_people_df):,} rows")
print(f"  By search_anchor_used:")
print(all_people_df["search_anchor_used"].value_counts().to_string())
print(f"  Has board_company:   {all_people_df['board_company'].notna().sum():,}")
print(f"  Has primary_company: {all_people_df['primary_company'].notna().sum():,}")
all_people_df.head(3)

In [ ]:
# Cell 9 — normalise URLs for join
all_people_df["clean_url"] = all_people_df["linkedin_url"].apply(clean_url)

In [ ]:
# Cell 10 — confirmation columns, ALIAS-AWARE board leg + a within-run NO-ALIAS baseline.
# The pure alias effect is, BY CONSTRUCTION, a clean superset (same picked Revelio user):
#   movers = strong_match_either & ~strong_match_either_noalias
# board now tests the firm's gvkey alias set (Cell 6b); primary leg is unchanged.

revelio_url_match = []
revelio_name_confirmed = []
revelio_user_id_col = []

rev_co_conf_board = []           # alias-on board (main)
rev_co_conf_board_noalias = []   # original single-name board (baseline)
rev_co_conf_board_fuzzy = []
rev_co_conf_primary = []
rev_co_conf_primary_fuzzy = []
rev_co_conf_either = []          # alias-on either (main)
rev_co_conf_either_noalias = []  # alias-off either (baseline)
rev_co_conf_either_fuzzy = []

for row in all_people_df.itertuples(index=False):
    clean = getattr(row, "clean_url", None)
    rev = revelio_by_url.loc[clean] if (clean and clean in revelio_by_url.index) else None

    if rev is None:
        revelio_url_match.append(False)
        revelio_name_confirmed.append(False)
        revelio_user_id_col.append(None)
        for L in (rev_co_conf_board, rev_co_conf_board_noalias, rev_co_conf_board_fuzzy,
                  rev_co_conf_primary, rev_co_conf_primary_fuzzy,
                  rev_co_conf_either, rev_co_conf_either_noalias, rev_co_conf_either_fuzzy):
            L.append(False)
        continue

    revelio_url_match.append(True)
    uid = rev["user_id"]
    revelio_user_id_col.append(uid)
    revelio_name_confirmed.append(
        name_matches(rev["fullname"], row.person_name, getattr(row, "person_name_clean", None))
    )

    board_co = getattr(row, "board_company", None)
    primary_co = getattr(row, "primary_company", None)

    _g = norm_gvkey(getattr(row, "gvkey", None))
    board_set = set(alias_by_gvkey.get(_g, set()))
    _bc = strip_legal(board_co)
    if _bc and not pd.isna(board_co) and len(str(board_co)) >= MIN_COMPANY_NAME_LEN:
        board_set.add(_bc)

    b_alias   = company_in_positions_aliased(uid, board_set)
    b_noalias = company_in_positions(uid, board_co, board_co)
    p_strict  = company_in_positions(uid, primary_co, primary_co) if not pd.isna(primary_co) else False
    rev_co_conf_board.append(b_alias)
    rev_co_conf_board_noalias.append(b_noalias)
    rev_co_conf_primary.append(p_strict)
    rev_co_conf_either.append(b_alias or p_strict)
    rev_co_conf_either_noalias.append(b_noalias or p_strict)

    b_fuzzy = company_in_positions_aliased_fuzzy(uid, board_set)
    p_fuzzy = company_in_positions_fuzzy(uid, primary_co, primary_co) if not pd.isna(primary_co) else False
    rev_co_conf_board_fuzzy.append(b_fuzzy)
    rev_co_conf_primary_fuzzy.append(p_fuzzy)
    rev_co_conf_either_fuzzy.append(b_fuzzy or p_fuzzy)

all_people_df["revelio_url_match"] = revelio_url_match
all_people_df["revelio_name_confirmed"] = revelio_name_confirmed
all_people_df["revelio_user_id"] = revelio_user_id_col
all_people_df["revelio_company_confirmed_board"] = rev_co_conf_board
all_people_df["revelio_company_confirmed_board_noalias"] = rev_co_conf_board_noalias
all_people_df["revelio_company_confirmed_primary"] = rev_co_conf_primary
all_people_df["revelio_company_confirmed_either"] = rev_co_conf_either
all_people_df["revelio_company_confirmed_either_noalias"] = rev_co_conf_either_noalias
all_people_df["revelio_company_confirmed_board_fuzzy"] = rev_co_conf_board_fuzzy
all_people_df["revelio_company_confirmed_primary_fuzzy"] = rev_co_conf_primary_fuzzy
all_people_df["revelio_company_confirmed_either_fuzzy"] = rev_co_conf_either_fuzzy
all_people_df["revelio_company_confirmed"] = rev_co_conf_board
all_people_df["revelio_company_confirmed_fuzzy"] = rev_co_conf_board_fuzzy

verified_bool = all_people_df["verified"].fillna(False).astype(bool).tolist()
def _strong(co_list):
    return [(n or v) and c for n, v, c in zip(revelio_name_confirmed, verified_bool, co_list)]
all_people_df["strong_match_board"] = _strong(rev_co_conf_board)
all_people_df["strong_match_primary"] = _strong(rev_co_conf_primary)
all_people_df["strong_match_either"] = _strong(rev_co_conf_either)
all_people_df["strong_match_either_noalias"] = _strong(rev_co_conf_either_noalias)
all_people_df["strong_match_either_fuzzy"] = _strong(rev_co_conf_either_fuzzy)
all_people_df["strong_match"] = all_people_df["strong_match_board"]

_mv = pd.Series(all_people_df["strong_match_either"]).astype(bool) & ~pd.Series(all_people_df["strong_match_either_noalias"]).astype(bool)
print(f"Done. strong_match_either alias-on {sum(all_people_df['strong_match_either']):,} "
      f"vs alias-off {sum(all_people_df['strong_match_either_noalias']):,} "
      f"| pure alias movers (rows): {int(_mv.sum()):,}")


In [ ]:
# Cell 11 — summary stats: Tier 1 (board), primary-only, Tier 2 (either), fuzzy variants
# Plus director-only breakdown stratified by search_anchor_used.

verified = all_people_df["verified"].fillna(False).astype(bool).tolist()

def count_strong(name_conf_list, co_conf_list, verified_list):
    return sum((n or v) and c for n, v, c in zip(name_conf_list, verified_list, co_conf_list))

total = len(all_people_df)
found = all_people_df["clean_url"].notna().sum()
matched = sum(revelio_url_match)
name_conf = sum(revelio_name_confirmed)

print(f"Total rows:                        {total:>8,}")
print(f"Has URL:                           {found:>8,}")
print(f"Revelio URL match:                 {matched:>8,}  ({matched/found*100:.1f}% of found)")
print(f"  Name confirmed:                  {name_conf:>8,}  ({name_conf/matched*100:.1f}%)")
print()
print(f"Company confirmation rate (of Revelio-matched):")
for label, col in [
    ("BOARD strict",   rev_co_conf_board),
    ("BOARD fuzzy",    rev_co_conf_board_fuzzy),
    ("PRIMARY strict", rev_co_conf_primary),
    ("PRIMARY fuzzy",  rev_co_conf_primary_fuzzy),
    ("EITHER strict",  rev_co_conf_either),
    ("EITHER fuzzy",   rev_co_conf_either_fuzzy),
]:
    n = sum(col)
    print(f"  {label:<16} {n:>8,}  ({n/matched*100:.1f}%)")
print()
print(f"Strong match rate (of Revelio-matched):")
for label, col in [
    ("Tier 1 (board strict)",   rev_co_conf_board),
    ("Tier 1 (board fuzzy)",    rev_co_conf_board_fuzzy),
    ("Tier 2 (either strict)",  rev_co_conf_either),
    ("Tier 2 (either fuzzy)",   rev_co_conf_either_fuzzy),
]:
    s = count_strong(revelio_name_confirmed, col, verified)
    print(f"  {label:<26} {s:>8,}  ({s/matched*100:.1f}%)")
print()

# ── Director-only breakdown stratified by search_anchor_used ──
print("=" * 70)
print("Director-only breakdown (source contains 'director' OR 'def14a')")
print("=" * 70)

source_str = all_people_df["source"].fillna("").astype(str)
is_director_row = source_str.str.contains("director|def14a", case=False, regex=True)

for anchor_label, anchor_mask in [
    ("ALL directors",     is_director_row),
    ("Original (board)",  is_director_row & (all_people_df["search_anchor_used"] == "board")),
    ("DEF 14A (primary)", is_director_row & (all_people_df["search_anchor_used"] == "primary")),
]:
    idx = anchor_mask[anchor_mask].index.tolist()
    if not idx:
        print(f"\n{anchor_label}: 0 rows")
        continue
    n_rows = len(idx)
    n_matched = sum(revelio_url_match[i] for i in idx)
    if n_matched == 0:
        print(f"\n{anchor_label}: {n_rows:,} rows, 0 Revelio-matched")
        continue
    s_board  = count_strong([revelio_name_confirmed[i] for i in idx],
                            [rev_co_conf_board[i] for i in idx],
                            [verified[i] for i in idx])
    s_either = count_strong([revelio_name_confirmed[i] for i in idx],
                            [rev_co_conf_either[i] for i in idx],
                            [verified[i] for i in idx])
    s_either_fuzzy = count_strong([revelio_name_confirmed[i] for i in idx],
                                  [rev_co_conf_either_fuzzy[i] for i in idx],
                                  [verified[i] for i in idx])
    print(f"\n{anchor_label}: {n_rows:,} rows, {n_matched:,} Revelio-matched")
    print(f"  Tier 1 strong (board strict):      {s_board:,}  ({s_board/n_matched*100:.1f}% of matched)")
    print(f"  Tier 2 strong (either strict):     {s_either:,}  ({s_either/n_matched*100:.1f}% of matched)")
    print(f"  Tier 2 strong (either fuzzy):      {s_either_fuzzy:,}  ({s_either_fuzzy/n_matched*100:.1f}% of matched)")
    print(f"  Δ Tier2 − Tier1 (strict):          {s_either - s_board:+,}")

In [ ]:
import re as _re
import unicodedata as _ud

_CREDS = _re.compile(r"\b(jr|sr|ii|iii|iv|cpa|mba|md|phd|esq|hon|dds|mph|edd|ms|ma)\b\.?", _re.I)
_DOTTED = _re.compile(r"\b[a-z]+(?:\.[a-z]+)+\.?", _re.I)
_PUNCT = _re.compile(r"[^\w\s]")
_WS = _re.compile(r"\s+")

def _normalize_name(name):
    """Return 'first last' normalized: ASCII-folded, lowercased, credentials and
    dotted-initial sequences stripped, single-letter middle/trailing tokens
    dropped. Used as the person component of the seat key."""
    if not isinstance(name, str):
        return ""
    folded = _ud.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    s = _DOTTED.sub(" ", folded.lower().strip())
    s = _CREDS.sub("", s)
    s = _WS.sub(" ", _PUNCT.sub(" ", s)).strip()
    if not s:
        return ""
    parts = s.split()
    if len(parts) == 1:
        return parts[0]
    kept = [parts[0]] + [p for p in parts[1:] if len(p) > 1]
    return kept[0] if len(kept) == 1 else f"{kept[0]} {kept[-1]}"

_ticker_str = all_people_df["ticker"].fillna("").astype(str).str.strip()
_seat_norm = all_people_df["person_name"].apply(_normalize_name) + "|" + _ticker_str
# Check against ALL WRDS sources (directors, executives, blockholders), not just directors
# A seat is def14a_only=True if it appears ONLY in def14a_serper rows, never in WRDS data
_source_str = all_people_df["source"].fillna("").astype(str)
_wrds_mask = _source_str.str.contains("director|executive|blockholder", case=False, regex=True)
_wrds_seats = set(_seat_norm[_wrds_mask])
all_people_df["def14a_only"] = ~_seat_norm.isin(_wrds_seats)
print(f"def14a_only flag set: {int(all_people_df['def14a_only'].sum()):,} rows "
      f"({all_people_df['def14a_only'].mean()*100:.1f}%)")

In [ ]:
# Cell 13 — S&P 500 coverage analysis
# NOTE: v2 dataset has no `is_entity` column. Filter is skipped (treat all rows as people).
sp500_table = redivis.user("ml2068").dataset("sp500").table("sp500_companies")
sp500_df = sp500_table.to_pandas_dataframe(variables=["gvkey", "ticker", "company_name"])

def norm_gvkey(x):
    try:
        return str(int(float(x)))
    except (ValueError, TypeError):
        return None

sp500_gvkeys = set(norm_gvkey(g) for g in sp500_df["gvkey"].dropna())
sp500_gvkeys.discard(None)
print(f"S&P 500 companies: {len(sp500_gvkeys):,} unique gvkeys")

# Normalise gvkeys + tag S&P 500
all_people_df["gvkey_norm"] = all_people_df["gvkey"].apply(norm_gvkey)
all_people_df["is_sp500"] = all_people_df["gvkey_norm"].isin(sp500_gvkeys)

# Signal columns (already computed in Cell 10/12 — re-bind for clarity)
all_people_df["revelio_url_match_col"] = revelio_url_match
all_people_df["verified_bool"] = all_people_df["verified"].fillna(False).astype(bool)

# v2 dataset has no is_entity; treat all rows as people
if "is_entity" in all_people_df.columns:
    people_df = all_people_df[all_people_df["is_entity"] == False].copy()
else:
    people_df = all_people_df.copy()
print(f"People rows: {len(people_df):,}")

sp500_p = people_df[people_df["is_sp500"]]
non_sp500_p = people_df[~people_df["is_sp500"]]

# ── Helper ──────────────────────────────────────────────────────────────
def coverage_stats(df, label):
    total = len(df)
    has_url = df["linkedin_url"].notna().sum()
    verified = df["verified_bool"].sum()
    rev_match = df["revelio_url_match_col"].sum()
    s_board = df["strong_match_board"].sum()
    s_either = df["strong_match_either"].sum()
    print(f"{label} (n={total:,}):")
    print(f"  Has URL:            {has_url:>8,}  ({has_url/total*100:.1f}% of people)")
    print(f"  Verified (name):    {verified:>8,}  ({verified/total*100:.1f}% of people)")
    print(f"  Revelio matched:    {rev_match:>8,}  ({rev_match/has_url*100:.1f}% of URLs found)")
    if rev_match:
        print(f"  Tier 1 strong:      {s_board:>8,}  ({s_board/rev_match*100:.1f}% of Revelio matched)")
        print(f"  Tier 2 strong:      {s_either:>8,}  ({s_either/rev_match*100:.1f}% of Revelio matched)")
    print()

coverage_stats(sp500_p,     "S&P 500 companies")
coverage_stats(non_sp500_p, "Non-S&P 500 companies")
coverage_stats(people_df,   "All people")

# ── Source × anchor breakdown ─────────────────────────────────────────────
print("Strong match rate by source × search_anchor_used (Tier 1 / Tier 2):")
for src, grp in people_df.groupby("source"):
    for anchor, sub in grp.groupby("search_anchor_used"):
        rev = sub["revelio_url_match_col"].sum()
        s_b = sub["strong_match_board"].sum()
        s_e = sub["strong_match_either"].sum()
        rate_b = s_b / rev * 100 if rev else 0
        rate_e = s_e / rev * 100 if rev else 0
        print(f"  {str(src)[:30]:<30} anchor={anchor:<8} n={len(sub):>6,}  "
              f"T1={s_b:>5,} ({rate_b:.1f}%)  T2={s_e:>5,} ({rate_e:.1f}%)")
print()

In [ ]:
# Cell 14 — PER-DIRECTOR-SEAT ROLLUP (headline number)
# Each (normalized person_name, ticker) is one director-board seat — matches the
# baseline unit (n≈41,591) reported in revelio_crosscheck.md.
# A director-seat is strong-matched if ANY of its URL rows (original board-anchored
# OR new DEF 14A primary-anchored) passes the rule.
#
# Name normalization (mirrors src/revelio/normalize_names.py — kept inline so this
# cell runs on Redivis without external imports). Without it, the same (person,
# ticker) gets counted twice when WRDS and DEF 14A spell the name differently
# (middle initials, honorifics, accents). Inflated raw counts on 2026-05-18 were
# ~15% high; correcting gives an honest headline.

import re as _re
import unicodedata as _ud

_CREDS = _re.compile(r"\b(jr|sr|ii|iii|iv|cpa|mba|md|phd|esq|hon|dds|mph|edd|ms|ma)\b\.?", _re.I)
_DOTTED = _re.compile(r"\b[a-z]+(?:\.[a-z]+)+\.?", _re.I)
_PUNCT = _re.compile(r"[^\w\s]")
_WS = _re.compile(r"\s+")

def _normalize_name(name):
    """Return 'first last' normalized: ASCII-folded, lowercased, credentials and
    dotted-initial sequences stripped, single-letter middle/trailing tokens
    dropped. Used as the person component of the seat key."""
    if not isinstance(name, str):
        return ""
    folded = _ud.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    s = _DOTTED.sub(" ", folded.lower().strip())
    s = _CREDS.sub("", s)
    s = _WS.sub(" ", _PUNCT.sub(" ", s)).strip()
    if not s:
        return ""
    parts = s.split()
    if len(parts) == 1:
        return parts[0]
    kept = [parts[0]] + [p for p in parts[1:] if len(p) > 1]
    return kept[0] if len(kept) == 1 else f"{kept[0]} {kept[-1]}"


source_str = all_people_df["source"].fillna("").astype(str)
is_director_row = source_str.str.contains("director|def14a", case=False, regex=True)
dir_df = all_people_df[is_director_row].copy()

# Force pandas-native bool dtype (Redivis uses pyarrow which can break .sum())
for col in ["revelio_url_match", "strong_match_board", "strong_match_primary",
            "strong_match_either", "strong_match_either_fuzzy"]:
    if col in dir_df.columns:
        dir_df[col] = dir_df[col].fillna(False).astype(bool)
dir_df["has_url_bool"] = dir_df["linkedin_url"].notna().astype(bool)

# Seat identity: (normalized first+last person name, ticker)
dir_df["seat_key"] = (
    dir_df["person_name"].apply(_normalize_name) + "|" +
    dir_df["ticker"].fillna("").astype(str)
)
# Keep the raw key around for reporting the dedup magnitude
dir_df["seat_key_raw"] = (
    dir_df["person_name"].fillna("").astype(str).str.lower().str.strip() + "|" +
    dir_df["ticker"].fillna("").astype(str)
)

per_seat = dir_df.groupby("seat_key").agg(
    has_url             = ("has_url_bool", "max"),
    has_revelio_match   = ("revelio_url_match", "max"),
    strong_board        = ("strong_match_board", "max"),
    strong_primary      = ("strong_match_primary", "max"),
    strong_either       = ("strong_match_either", "max"),
    strong_either_fuzzy = ("strong_match_either_fuzzy", "max"),
).reset_index()

# Convert to numpy bool/int for clean summing
for c in ["has_url", "has_revelio_match", "strong_board", "strong_primary",
          "strong_either", "strong_either_fuzzy"]:
    per_seat[c] = per_seat[c].astype(bool).astype(int)

n_raw_seats = dir_df["seat_key_raw"].nunique()
n_seats = len(per_seat)
n_with_url = int(per_seat["has_url"].sum())
n_rev_matched = int(per_seat["has_revelio_match"].sum())
n_t1 = int(per_seat["strong_board"].sum())
n_primary = int(per_seat["strong_primary"].sum())
n_t2 = int(per_seat["strong_either"].sum())
n_t2f = int(per_seat["strong_either_fuzzy"].sum())

def pct(num, den):
    return (num / den * 100) if den else 0.0

print("=" * 70)
print("PER-DIRECTOR-SEAT ROLLUP   (unit: unique normalized person × ticker)")
print("=" * 70)
print(f"Director-seats (normalized):  {n_seats:,}")
print(f"  (raw, pre-normalization:    {n_raw_seats:,}  — {n_raw_seats - n_seats:,} dupes collapsed)")
print(f"  With URL:                  {n_with_url:>8,}  ({pct(n_with_url, n_seats):.1f}% of seats)")
print(f"  Revelio-matched:           {n_rev_matched:>8,}  ({pct(n_rev_matched, n_seats):.1f}% of seats)")
print()
print(f"Strong matches per seat (counted if ANY URL row passes):")
print(f"  Tier 1 (board strict):     {n_t1:>8,}  "
      f"({pct(n_t1, n_rev_matched):.1f}% of Revelio-matched, {pct(n_t1, n_seats):.1f}% of all seats)")
print(f"  Primary-only (strict):     {n_primary:>8,}  ({pct(n_primary, n_rev_matched):.1f}% of Revelio-matched)")
print(f"  Tier 2 (either strict):    {n_t2:>8,}  "
      f"({pct(n_t2, n_rev_matched):.1f}% of Revelio-matched, {pct(n_t2, n_seats):.1f}% of all seats)")
print(f"  Tier 2 (either fuzzy):     {n_t2f:>8,}  "
      f"({pct(n_t2f, n_rev_matched):.1f}% of Revelio-matched, {pct(n_t2f, n_seats):.1f}% of all seats)")
print()
print(f"BASELINE comparison (revelio_crosscheck.md, normalized):")
print(f"  Baseline strong match: 8,809 (34.2% of Revelio-matched directors, n≈41,560)")
print(f"  NEW Tier 1:            {n_t1:,} ({pct(n_t1, n_rev_matched):.1f}%)  — sanity reproduction")
print(f"  NEW Tier 2 (either):   {n_t2:,} ({pct(n_t2, n_rev_matched):.1f}%)  — Δ +{n_t2 - 8809:,} seats (+{pct(n_t2, n_rev_matched) - 34.2:.1f}pp)")
print(f"  NEW Tier 2 (fuzzy):    {n_t2f:,} ({pct(n_t2f, n_rev_matched):.1f}%)  — Δ +{n_t2f - 8809:,} seats (+{pct(n_t2f, n_rev_matched) - 34.2:.1f}pp)")


In [ ]:
# Cell 15 — Export validation summary dataset
# Write all_people_df with validation results as output table (downloadable from Redivis)

output_cols = [
    "linkedin_url",
    "person_name",
    "person_name_clean",
    "board_company",
    "primary_company",
    "company_name_clean",
    "source",
    "search_anchor_used",
    "ticker",
    "gvkey",
    "revelio_url_match",
    "revelio_name_confirmed",
    "revelio_company_confirmed_board",
    "revelio_company_confirmed_primary",
    "revelio_company_confirmed_either",
    "revelio_company_confirmed_board_fuzzy",
    "revelio_company_confirmed_primary_fuzzy",
    "revelio_company_confirmed_either_fuzzy",
    "revelio_user_id",
    "strong_match_board",
    "strong_match_primary",
    "strong_match_either",
    "strong_match_either_noalias",
    "strong_match_either_fuzzy",
    "verified",
    "def14a_only",
]

output_df = all_people_df[output_cols].copy()
print(f"Exporting validation summary: {len(output_df):,} rows × {len(output_cols)} columns")

# Write to CSV file
output_df.to_csv("revelio_validation_summary_v3.csv", index=False)
print(f"✓ Written CSV: revelio_validation_summary_v3.csv")

# Create Redivis output table (downloadable from Redivis UI)
redivis.current_notebook().create_output_table(output_df)
print(f"✓ Output table created in Redivis workflow")

In [ ]:
# Cell 16b — CLEAN alias movers = within-run (alias-on) minus (alias-off).  [v3 NEW]
# Guaranteed superset: same picked Revelio user, so the ONLY difference is the alias set.
mask = pd.Series(all_people_df["strong_match_either"]).astype(bool) & \
       ~pd.Series(all_people_df["strong_match_either_noalias"]).astype(bool)
movers = all_people_df[mask].copy()
mcols = ["linkedin_url","person_name","board_company","primary_company","ticker","gvkey",
         "revelio_user_id","strong_match_either_noalias","strong_match_either"]
movers[mcols].to_csv("alias_movers_v3.csv", index=False)
print(f"Alias movers — distinct URLs: {movers['linkedin_url'].nunique():,}  (rows: {len(movers):,})")
print("Top firms gaining strong matches:")
print(movers["board_company"].value_counts().head(15).to_string())
print()
print("Pichai present:", movers["person_name"].str.contains("pichai", case=False, na=False).any())
# (movers are also derivable from the main export via strong_match_either & ~strong_match_either_noalias)


In [ ]:
# Cell 16 — CHECK: does Revelio have age / birth-year / education (age-proxy) fields?
# The cells above only SELECT a subset of columns, so an age/birth field could
# exist and just never be pulled. This dumps the FULL column schema of every
# Revelio table we use and flags age-ish columns. Run on Redivis.

AGE_KEYS = ["age", "birth", "dob", "born", "yob", "education", "educ",
            "degree", "grad", "school", "university", "college", "enroll"]

def schema_of(ref):
    """Return column list for a Revelio table ref via a 1-row query."""
    df = redivis.query(f"SELECT * FROM `{ref}` LIMIT 1").to_pandas_dataframe()
    return df.columns.tolist()

TABLE_REFS = [
    "individual_user:xcsm",
    "individual_position:8xgp",
]

for ref in TABLE_REFS:
    try:
        cols = schema_of(ref)
        hits = [c for c in cols if any(k in c.lower() for k in AGE_KEYS)]
        print(f"\n=== {ref}  ({len(cols)} columns) ===")
        print(cols)
        print("  -> age/birth/education-ish columns:", hits if hits else "NONE")
    except Exception as e:
        print(f"\n=== {ref} === ERROR: {e}")

# If the Revelio dataset has an education table (degree start/end years let you
# ESTIMATE age, bachelor start ~18), add its ref to TABLE_REFS above. Browse the
# dataset in the Redivis UI for a table like `individual_user_education`.
print("\nDone. If no age/birth column appears, the only Revelio age signal would be"
      "\neducation dates (an estimate, not a stated age).")
